# STAGATE Benchmark for spatial mutliomcis data integration on simulated dataset

Notebook benchmarks spatial mutliomcis data integration using STAGATE on simulated dataset.

## Loading

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import omicverse as ov
import anndata as ad
import pandas as pd
import scanpy as sc
import numpy as np

## STAGATE Pipeline

In [ ]:
data_dir = 'Original_Simulated_Data'
output_dir = 'Processed_Simulated_Data'
os.makedirs(output_dir, exist_ok=True)

# 循环处理每个数据集
for i in range(1, 6):
    
    print(f"Process {data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad data.")
          
    adata_rna = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad')
    adata_rna = adata_rna.raw.to_adata()
    adata_rna.obs['ground_truth'] = adata_rna.obs['cell_type'] 

    sc.pp.highly_variable_genes(adata_rna, n_top_genes=3000)
    adata_rna = adata_rna[:,adata_rna.var['highly_variable'] == True]

    methods_kwargs={}
    methods_kwargs['STAGATE']={
        'num_batch_x':1,'num_batch_y':1,
        'spatial_key':['X','Y'],'rad_cutoff':0.17,
        'num_epoch':1000,'lr':0.001,
        'weight_decay':1e-4,'hidden_dims':[512, 30],
        'device':'cuda:0',
        #'n_top_genes':2000,
    }

    # spatial clustering
    adata=ov.space.clusters(adata_rna,
                      methods=['STAGATE'],
                     methods_kwargs=methods_kwargs)

    ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['STAGATE'].shape[1],
                   use_rep='STAGATE')

    # leiden clustering
    ov.utils.cluster(adata_rna,use_rep='STAGATE',method='leiden',resolution=0.65
                    )

    # plot
    sc.pl.spatial(adata_rna,color=['ground_truth','leiden'],spot_size=0.12,wspace=0.4)

    # data saving
    adata_rna.write_h5ad(f'{output_dir}/Simulated_Dataset_{i}/stagate_rna.h5ad',compression='gzip')

In [ ]:
!pip list